## Purpose
Understand all raw datasets before building cleaning and transformation pipelines.

## Scope
- Identify structure, grain, and key fields
- Surface data quality issues
- Define cleaning rules

## Master Cleaning Rules

# Helper Functions

In [77]:
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', None)
import csv
import numpy

## get_sources
Obtain the sources of raw data used to analysis in this project, outputs a list of subfolders containing the name of the data source


In [46]:
def get_sources():
    """
    Return list of dataset folders in Data/RawData
    """
    current = Path.cwd()
    while not (current / "Data").exists():
        current = current.parent

    raw_path = current / "Data" / "RawData"
    
    return [
        folder.name for folder in raw_path.iterdir()
        if folder.is_dir()
    ]


## read_files
Obtain the raw tabular files (csv/xls/xlsx) files of sources used to analysis in this project, outputs a list of the names and extensions of the file

@param name of the source, obtained from get_sources function

In [47]:

def read_files(source):
    """
    Return list of readable file paths from Data/RawData/<source>
    """
    current = Path.cwd()
    while not (current / "Data").exists():
        current = current.parent

    data_path = current / "Data" / "RawData" / source
    
    return [
        file for file in data_path.iterdir()
        if file.suffix.lower() in [".csv", ".xlsx", ".xls"]
    ]

## pull_data
Reads the contents of the tabular data, and outputs it as a dataframe

@param name of the filesource, outputted from read_files
outputs a dictionary of all the files from the source

In [48]:
def pull_data(source):
    """
    Load all readable files from Data/RawData/<source>.

    Parameters:
    - source (str): folder name under RawData (e.g., 'ward')

    Returns:
    - dict: {file_name_without_extension: DataFrame}
    """
    files = read_files(source)
    data = {}

    for file in files:
        file = Path(file)
        try:
            if file.suffix.lower() == ".csv":
                try:
                    df = pd.read_csv(file, on_bad_lines="skip", encoding="utf-8")
                except UnicodeDecodeError:
                    #print(f"⚠️ Encoding issue in {file.name}, retrying with latin1") #debugging
                    df = pd.read_csv(file, on_bad_lines="skip", encoding="latin1")
            elif file.suffix.lower() in [".xlsx", ".xls"]:
                df = pd.read_excel(file)
            else:
                continue
            data[file.stem] = df
            #print(f"Loaded: {file.name}")
        except Exception as e:
            print(f"\n❌ Error loading {file.name}")
            print(e)

            # 🔍 Debug: find problematic rows
            if file.suffix.lower() == ".csv":
                print("🔎 Scanning for bad rows...")

                with open(file, "r", encoding="utf-8") as f:
                    reader = csv.reader(f)
                    expected_len = len(next(reader))  # header length

                    for i, row in enumerate(reader, start=2):
                        if len(row) != expected_len:
                            print(f"\n⚠️ Bad row at line {i}:")
                            print(row)
                            break  # stop at first bad row
    return data

## full_summary or quick_summary
@Param data outputted by a specific dataset of the source

In [49]:
#Quick summary, or full summary, loads the dataframe and pulls the dtypes, % missing data, and # unique values
def quick_summary(df):
    return pd.DataFrame({
        "dtype": df.dtypes,
        "missing_pct": df.isnull().mean(),
        "n_unique": df.nunique()
    })

def full_summary(data):
    for name, df in data.items():
        print(f"\nSummary for: {name}")
        print(quick_summary(df))
    return

## compare_columns
@param data which contains the dictionary of all the data available for a source, outputted from pull_data

outputs a table showing which columns exist for a given dataset and which aren't from a union of all aggregatd columns for that source

In [50]:
def compare_columns(data):

    all_cols = sorted(set().union(*[df.columns for df in data.values()]))

    table = pd.DataFrame(index=all_cols)

    for name, df in data.items():
        table[name] = table.index.isin(df.columns)

    return table

## column_presence

@param a list containing all data sets from a source, outputter from pull_data

outputs a table showing the presence of an aggregate of all datasets from source from 0-1 as well as which datasets those columns are present in

if you want to prune datasets below a certain threshold, it can be done as so
scores = column_presence_score(data)
cols_to_keep = scores[scores["presence_score"] >= 0.8].index


In [51]:
## Show the presence of columns accross the entire dataset for a given source, to better understand which columns are common
def column_presence(data):
    """
    Return each column's presence score and the source tables it appears in.
    """
    all_cols = sorted(set().union(*[set(df.columns) for df in data.values()]))
    n_tables = len(data)

    rows = []
    for col in all_cols:
        present_in = [name for name, df in data.items() if col in df.columns]
        rows.append({
            "column": col,
            "presence_score": len(present_in) / n_tables,
            "present_in_tables": present_in
        })

    return pd.DataFrame(rows).sort_values(
        ["presence_score", "column"],
        ascending=[False, True]
    ).reset_index(drop=True)

In [52]:
##

# Dataset: 311 Service Requests - Customer Initiated

## Description
The dataset contains information on customer initiated service requests received by 311 Toronto. 
This data was collected from multiple channels including telephone, fax, email, online self-serve requests, mobile API and Twitter.

311 manages City divisional service request data for Solid Waste Management, Transportation Services, Toronto Water, Municipal Licensing & Standards,
and Urban Forestry. For our purposes, we will be focusing on Toronto Water


## Source
City Open Data Catalogue - CKAN API 
https://open.toronto.ca/dataset/311-service-requests-customer-initiated/
Index # is 0

## Files included
SR2010.csv - SR2026.csv

## Files excluded
311-service-requests-readme.xlsx

## Grain
One instance of a 311 service request

## Cleaning Rules

1. Remove non-data files (e.g., readme)

2. Standardize column names
   - convert to snake_case
   - lowercase, remove spaces/special characters

3. Align schema across datasets
   - ensure consistent column structure before combining

4. Handle missing values
   - assess missingness
   - decide to drop, impute, or retain

5. Enforce data types
   - convert date fields to datetime
   - ensure numeric fields are numeric
   - cast categorical fields appropriately

6. Standardize values
   - fix inconsistent categories (e.g., casing)
   - trim whitespace

7. Drop unnecessary columns
   - Remove all but Creation_date, Status, Postal, Ward, Type, Division, Section

8. Validate data
   - ensure key fields are not null
   - remove invalid records


## Initiate Data Exploration

In [53]:
#Read available datasets available for data source
DataSources = get_sources()
print(DataSources)

['311-service-requests-customer-initiated', 'current-and-future-climate', 'neighbourhoods', 'ward-profiles-25-ward-model', 'watermain-breaks', 'watermains']


In [54]:
source = (DataSources[0])
readable = read_files(source)
readable

[WindowsPath('C:/Users/1/Desktop/project/Data/RawData/311-service-requests-customer-initiated/311-service-requests-readme.xlsx'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/311-service-requests-customer-initiated/SR2010.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/311-service-requests-customer-initiated/SR2011.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/311-service-requests-customer-initiated/SR2012.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/311-service-requests-customer-initiated/SR2013.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/311-service-requests-customer-initiated/SR2014.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/311-service-requests-customer-initiated/SR2015.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/311-service-requests-customer-initiated/SR2016.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/311-service-requests-customer-initiated/SR2017.csv'),
 Windo

### Pull Data to view datasets and keys of datasets from source

In [55]:
data_311 = (pull_data(source))
print(f"number of datasets from source: {len(data_311)}")
print(f"names of remaining datasets: {data_311.keys()}")

number of datasets from source: 18
names of remaining datasets: dict_keys(['311-service-requests-readme', 'SR2010', 'SR2011', 'SR2012', 'SR2013', 'SR2014', 'SR2015', 'SR2016', 'SR2017', 'SR2018', 'SR2019', 'SR2020', 'SR2021', 'SR2022', 'SR2023', 'SR2024', 'SR2025', 'SR2026'])


### Run summary of the datasets for source

full_summary(data) for a full summary of all datasets, quick_summary(data) for source of singular dataset

In [56]:
full_summary(data_311)
#quick_summary(data_311['SR2026'])


Summary for: 311-service-requests-readme
                           dtype  missing_pct  n_unique
Field / Column Name       object          0.0         7
Description / Definition  object          0.0         7
Comments / Example        object          0.0         7

Summary for: SR2010
                               dtype  missing_pct  n_unique
Creation Date                 object     0.000000    251595
Status                        object     0.000000         6
First 3 Chars of Postal Code  object     0.000000        99
Intersection Street 1         object     0.929368      3114
Intersection Street 2         object     0.929533      2984
Ward                          object     0.000000        44
Service Request Type          object     0.000000       369
Division                      object     0.000000         7
Section                       object     0.000000        12

Summary for: SR2011
                               dtype  missing_pct  n_unique
Creation Date                 ob

### Compare columns for a quick view of how uniform the data is across source

In [57]:
compare_columns(data_311)

,311-service-requests-readme,SR2010,SR2011,SR2012,SR2013,SR2014,SR2015,SR2016,SR2017,SR2018,SR2019,SR2020,SR2021,SR2022,SR2023,SR2024,SR2025,SR2026
Comments / Example,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
Creation Date,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Description / Definition,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
Division,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Field / Column Name,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
First 3 Chars of Postal Code,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Intersection Street 1,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Intersection Street 2,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Section,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Service Request Type,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


### Check for how much presence columns have across all datasets

In [58]:
column_presence(data_311)
#check the unique values for a specific column

,column,presence_score,present_in_tables
0,Creation Date,0.944444,"[SR2010, SR2011, SR2012, SR2013, SR2014, SR201..."
1,Division,0.944444,"[SR2010, SR2011, SR2012, SR2013, SR2014, SR201..."
2,First 3 Chars of Postal Code,0.944444,"[SR2010, SR2011, SR2012, SR2013, SR2014, SR201..."
3,Intersection Street 1,0.944444,"[SR2010, SR2011, SR2012, SR2013, SR2014, SR201..."
4,Intersection Street 2,0.944444,"[SR2010, SR2011, SR2012, SR2013, SR2014, SR201..."
5,Section,0.944444,"[SR2010, SR2011, SR2012, SR2013, SR2014, SR201..."
6,Service Request Type,0.944444,"[SR2010, SR2011, SR2012, SR2013, SR2014, SR201..."
7,Status,0.944444,"[SR2010, SR2011, SR2012, SR2013, SR2014, SR201..."
8,Ward,0.944444,"[SR2010, SR2011, SR2012, SR2013, SR2014, SR201..."
9,Comments / Example,0.055556,[311-service-requests-readme]


# Climate Data

## Description
This data shows climate projections for 53 climate variables across two climate scenarios (SSP2-4.5 and SSP5-8.5) and three future time periods (2015-2040, 2041- 2070, 2071-2100). This dataset represents the averages across the City of Toronto's municipal boundary area. Climate variables covering temperature, precipitation, extreme heat and cold days, dry days, agricultural variables, freeze-thaw cycles, and freezing rain potential are presented. 

The five SSP pathways can be characterized in terms of the socioeconomic challenges they imply for mitigating and adapting to climate change:
-SSP1 (Sustainability) has few challenges to both mitigation and adaptation. In this pathway, policies focus on human well-being, clean energy technologies, and the preservation of the natural environment.

-SSP2 (Middle of the Road) represents moderate challenges to both mitigation and adaptation and follows a pathway of balanced energy development.

-SSP3 (Regional Rivalry) is characterized by many challenges to both mitigation and adaptation. In this pathway, which relies heavily on fossil fuels and an increased use of coal, nationalism drives policy and focus is placed on regional and local issues rather than on global issues. 

-SSP4 (Inequality), which follows a similar energy future to SSP1, is defined by many challenges to adaptation and few challenges to mitigation. 

-SSP5 (Fossil-fueled Development), also heavily reliant on fossil fuels including coal, is characterized by many challenges to mitigation and few challenges to adaptation

In addition to SSP, the IPICC Sixth Assessment Report combines SSPs with the expected level of radative Forcing in the year 2100, characterized by the following table:

SSP 	|Scenario	|Estimated warming  (2041–2060) 	|Estimated warming  (2081–2100) 	|Very likely range in °C  (2081–2100)|
--------|-----------|-----------------------------------|-----------------------------------|------------------------------------|
SSP1-1.9|very low GHG emissions: CO2 emissions cut to net zero around 2050 |	1.6 °C |	1.4 °C |	1.0 – 1.8|
SSP1-2.6| 	low GHG emissions: CO2 emissions cut to net zero around 2075 |1.7 °C 	|1.8 °C 	|1.3 – 2.4|
SSP2-4.5| 	intermediate GHG emissions: CO2 emissions around current levels until 2050, then falling but not reaching net zero by 2100| 	2.0 °C| 	2.7 °C |	2.1 – 3.5|
SSP3-7.0| 	high GHG emissions: CO2 emissions double by 2100| 	2.1 °C| 	3.6 °C| 	2.8 – 4.6|
SSP5-8.5| 	very high GHG emissions: CO2 emissions triple by 2075| 	2.4 °C| 	4.4 °C| 	3.3 – 5.7| 

This dataset simulates only for scenario SSP2-4.5 and SSP5-8.5

## Source
City Open Data Catalogue - CKAN API 
https://open.toronto.ca/dataset/current-and-future-climate/
Index # is 1 

Explanation for SSP and Radiative Forcing:
Understanding Shared Socio-economic Pathways (SSPs)
https://climatedata.ca/resource/understanding-shared-socio-economic-pathways-ssps/

IPCC Sixth Assessment Report
https://www.ipcc.ch/report/ar6/wg1/


## Files included
Readme
ServiceRequests csv files from 2010-2026

## Grain
One simulation at a given climate scenario(SSP) and percentile

## Cleaning Rules -- given the complexity of this database, will return later.

1. Standardize column names
   - convert to snake_case
   - lowercase, remove spaces/special characters

2. Handle missing values
   - assess missingness
   - decide to drop, impute, or retain
   - Initial exploration shows the missing values come from overall trends
     

3. Enforce data types
   - convert date fields to datetime
   - ensure numeric fields are numeric
   - cast categorical fields appropriately

4. Standardize values
   - fix inconsistent categories (e.g., casing)
   - trim whitespace

5. Drop unnecessary columns
   - placeholder, will evaluate after
  
6. Drop unnecessart rows
   - initial exploration suggests dropping overall trends

7. Validate data
   - ensure key fields are not null
   - remove invalid records

## Initiate Data Exploration

In [59]:
#Read available datasets available for data source
DataSources = get_sources()
print(DataSources) #


['311-service-requests-customer-initiated', 'current-and-future-climate', 'neighbourhoods', 'ward-profiles-25-ward-model', 'watermain-breaks', 'watermains']


In [60]:
source = (DataSources[1])
readable = read_files(source)
readable

[WindowsPath('C:/Users/1/Desktop/project/Data/RawData/current-and-future-climate/climate-variables.csv')]

### Pull Data to view datasets and keys of datasets from source

In [61]:
data_climate = (pull_data(source))
print(f"number of datasets from source: {len(data_climate)}")
print(f"names of remaining datasets: {data_climate.keys()}")

number of datasets from source: 1
names of remaining datasets: dict_keys(['climate-variables'])


### Run summary of the datasets for source

full_summary(data) for a full summary of all datasets, quick_summary(data) for source of singular dataset

since we only have one dataset from source, we do not need to run full summary

In [62]:
#full_summary(data_climate)
quick_summary(data_climate['climate-variables'])

,dtype,missing_pct,n_unique
_id,int64,0.000000,27
Climate Scenario,object,0.000000,3
Time Horizon,object,0.074074,4
Distribution,object,0.000000,4
ANNUAL_MEAN_TEMPERATURE,object,0.000000,19
WINTER_MEAN_TEMPERATURE,object,0.000000,20
SPRING_MEAN_TEMPERATURE,object,0.000000,19
SUMMER_MEAN_TEMPERATURE,object,0.000000,20
FALL_MEAN_TEMPERATURE,object,0.000000,20
ANNUAL_MAXIMUM_TEMPERATURE,object,0.000000,19


In [63]:
data_climate['climate-variables']

,_id,Climate Scenario,Time Horizon,Distribution,ANNUAL_MEAN_TEMPERATURE,WINTER_MEAN_TEMPERATURE,SPRING_MEAN_TEMPERATURE,SUMMER_MEAN_TEMPERATURE,FALL_MEAN_TEMPERATURE,ANNUAL_MAXIMUM_TEMPERATURE,WINTER_MAXIMUM_TEMPERATURE,SPRING_MAXIMUM_TEMPERATURE,SUMMER_MAXIMUM_TEMPERATURE,FALL_MAXIMUM_TEMPERATURE,ANNUAL_MINIMUM_TEMPERATURE,WINTER_MINIMUM_TEMPERATURE,SPRING_MINIMUM_TEMPERATURE,SUMMER_MINIMUM_TEMPERATURE,FALL_MINIMUM_TEMPERATURE,DAYS_ABOVE_35C,DAYS_ABOVE_30C,DAYS_ABOVE_25C,DAYS_ABOVE_20C_TROPICAL_NIGHTS,HOTTEST_DAY_TEMPERATURE,TEMPERATURE_BASED_HEAT_WARNING_FREQUENCY,MAXIMUM_CONSECUTIVE_TEMPERATURE_BASED_HEAT_WARNING_DAYS,HUMIDEX_ABOVE_30,HUMIDEX_ABOVE_35,HUMIDEX_ABOVE_40,DAYS_BELOW_MINUS_20C,DAYS_BELOW_MINUS_10C,DAYS_BELOW_0C_FROST_DAYS,HEATING_DEGREE_DAYS,COOLING_DEGREE_DAYS,ANNUAL_TOTAL_PRECIPITATION,WINTER_TOTAL_PRECIPITATION,SPRING_TOTAL_PRECIPITATION,SUMMER_TOTAL_PRECIPITATION,FALL_TOTAL_PRECIPITATION,MAXIMUM_1DAY_PRECIPITATION,MAXIMUM_3DAY_PRECIPITATION,SIMPLE_DAILY_INTENSITY_INDEX,95TH_PERCENTILE_PRECIPITATION,99TH_PERCENTILE_PRECIPITATION,MAXIMUM_CONSECUTIVE_WET_DAYS,ANNUAL_TOTAL_DRY_DAYS,MAXIMUM_CONSECUTIVE_DRY_DAYS,FROST_FREE_SEASON_START_DATE,FROST_FREE_SEASON_END_DATE,FROST_FREE_SEASON_LENGTH,CORN_HEAT_UNITS,GROWING_DEGREE_DAYS_BASE_0C,CANOLA_GROWING_DEGREE_DAYS_BASE_4C,FORAGE_CROPS_GROWING_DEGREE_DAYS_BASE_5C,CORN_AND_BEAN_GROWING_DEGREE_DAYS_BASE_10C,GROWING_DEGREE_DAYS_BASE_15C,FREEZE_THAW_CYCLES,FREEZING_RAIN_POTENTIAL
0,1,OBSERVED_TORONTO_AVERAGE,1971-2000,MEDIAN,7.9,-4.3,6.5,19.7,9.5,12.5,-0.4,11.3,25.1,13.8,3.4,-7.9,1.7,14.4,5.3,0.3,9.9,57.8,5,33.1,0.5,1.2,-,-,-,3.7,39.2,133.7,3933.9,256.7,753.4,156.8,189.3,197.6,208.3,37.3,50.2,4.9,11.2,21.9,8,211,13.4,05-Apr,19-Nov,233,3602.7,3426.4,2422.4,2200.5,1248.3,548,65.5,3.2
1,2,SSP2-4.5,1971-2000,10th_PERCENTILE,7.2,-6.3,4.9,18.9,8.7,11.7,-2.5,9.4,24,12.8,2.7,-10.1,0.4,13.7,4.5,0,3.5,43.1,1.7,31.3,0,0,28.7,3.3,0,0.2,21.6,113.7,3596.8,194.1,671.9,125.6,140.9,141.5,144.9,28.5,41.1,4.4,9.9,18.9,4.2,193.8,8.9,20-Mar,03-Nov,208,3376.7,3247.4,2273.4,2058.8,1138.1,464.9,49.8,0.4
2,3,SSP2-4.5,1971-2000,MEDIAN,8.1,-4.1,6.5,19.9,9.9,12.5,-0.5,11.2,25.1,14.1,3.6,-7.7,1.9,14.6,5.6,0,9.9,57.6,5.5,33.4,0.2,0.6,41.1,8.7,0.2,2.6,36.3,129.8,3887.6,269.9,795.1,171.3,199,204,207.6,37.4,55.2,5.1,11.9,22.8,5.4,207.2,11.9,05-Apr,18-Nov,230,3631.4,3465,2453.5,2231,1274.2,568.6,61.9,2.1
3,4,SSP2-4.5,1971-2000,90th_PERCENTILE,9,-1.9,8.2,21,11.2,13.5,1.5,13.2,26.5,15.5,4.6,-5.2,3.4,15.6,6.9,1.1,20.1,72.1,11.9,35.5,2.2,4.8,52.9,17.4,2,7.4,51.4,145.1,4173.5,357.5,919.3,227,265.3,277.5,286.4,60.8,80.6,5.7,13.8,27.7,7.1,220.1,16.9,16-Apr,02-Dec,268,3927.1,3706.6,2658.5,2425.5,1421.3,680.2,74.5,4.4
4,5,SSP2-4.5,2015-2040,10th_PERCENTILE,9.1,-3.8,6.8,20.6,10.5,13.4,-0.4,11.3,25.7,14.6,4.7,-7.3,2.2,15.4,6.2,0,11.5,66,7.4,33,0.2,0.5,49.4,11,0.2,0,7.5,87.5,3010,333.5,700.6,139.6,149.2,140.4,148.8,29.4,42.9,4.6,10.4,19.7,4.4,193.2,9.1,07-Mar,13-Nov,227.2,3907.2,3716.1,2658.3,2423.9,1417.4,663.3,41.9,0.4
5,6,SSP2-4.5,2015-2040,MEDIAN,10.1,-1.7,8.4,21.7,11.8,14.5,1.5,13.2,27.1,16.1,5.6,-5,3.7,16.3,7.4,1.1,23.9,83.1,15,35.4,1.8,5.1,63.1,22.1,2,0.2,18.7,106.5,3324.2,441.2,832.8,188,212.1,208.7,213.1,40.4,58.2,5.3,12.3,24.1,5.6,207.5,12,24-Mar,26-Nov,260.2,4250.4,3995.4,2899.8,2656.7,1600,798,54.5,2.3
6,7,SSP2-4.5,2015-2040,90th_PERCENTILE,11.2,0.3,10.2,23,13.3,15.7,3.3,15.2,28.7,18,6.7,-2.7,5.2,17.4,8.9,6.5,40.2,101,26.5,38.2,4.3,14.2,78.5,34.2,6.8,2.6,32.9,124.9,3629.2,582.6,989.5,256,282.9,293.9,294.8,63,86.7,6.2,14.8,29.6,7.6,220.3,16.8,08-Apr,14-Dec,314.9,4590.2,4293.9,3173.4,2923.7,1821.9,972.3,68.1,5.3
7,8,SSP2-4.5,2041-2070,10th_PERCENTILE,10.1,-2.7,7.8,21.5,11.4,14.4,0.6,12.2,26.6,15.6,5.7,-6,3.3,16.3,7.1,0.2,18.9,77.9,14,34.1,1.5,3.9,62,19.6,1.1,0,2.2,68.5,2618.5,441.3,733.9,140.5,158.9,138.3,152.1,31.8,46.6,4.8,10.7,20.8,4.4,192.5,9,28-Feb,16-Nov,243,4222,3994.1,2897.2,2650.1,1583.6,780.7,35,0.6
8,9,SSP2-4.5,2041-2070,MEDIAN,11.3,-0.4,9.5,22.9,12.8,15.7,2.6,14.3,28.3,17.2,6.8,-3.3,4.7,17.4,8.4

# Neighbourhoods Data

## Description
Toronto’s 158 neighbourhoods are a microcosm of the city’s inhabitants, cultures and life. The primary purpose of the City-designated social planning neighbourhoods is to help City staff collect data, plan, analyze and forecast City services. While these neighbourhoods may not fully encompass every historical neighbourhood area, they do provide a way for planners and researchers to track information about them over time.

The neighbourhood profiles were developed to help government and community agencies with their local planning, by providing socio-economic data at a meaningful geographic area. Unlike other geographies like wards or dissemination blocks, the boundaries of these social planning neighbourhoods change very infrequently over time, allowing researchers to perform longitudinal studies to see the changes in each area. Not all people define neighbourhoods the same way, but for the purposes of statistical reporting these neighbourhoods were defined based on Statistics Canada census tracts.

Some neighbourhoods are officially designated as Neighbourhood Improvement Areas (NIA)

## Source
City Open Data Catalogue - CKAN API 
https://open.toronto.ca/dataset/neighbourhoods/
Index # is 2

Neighbourhood Equity Index:
https://www.toronto.ca/wp-content/uploads/2017/11/97eb-TSNS-2020-NEI-equity-index-methodology-research-report-backgroundfile-67350.pdf


## Additional Sources Considered

- Appendix III: Housing Suitability (TSNS 2020 report)

Notes:
- Potentially relevant to socioeconomic analysis
- No direct dataset source identified
- Not included in current pipeline due to unclear provenance
- May be revisited in future iterations

## Files included
TSNS 2020 report PDF which includes table in appendix 2
Neighbourhoods and Neighbourhoods historical.

## Files excluded
Neighbourhoods fields.csv
Neighbourhoods - historical 140 fields.csv

## Grain
One neighbourhood in the greater toronto area as well as any data associated with it (socioeconomic, boundaries, etc)

## Cleaning Rules -- given the complexity of this database, will return later.

1. Drop unnecessary datasets
   - drop all historical tables, we only need the most recent
   - drop fields tables

2. Standardize column names
   - convert to snake_case
   - lowercase, remove spaces/special characters
   - 
3. Handle missing values
   - assess missingness
   - decide to drop, impute, or retain
   - Initial exploration shows the missing values come from overall trends

4. Enforce data types
   - convert date fields to datetime
   - ensure numeric fields are numeric
   - cast categorical fields appropriately

5. Standardize values
   - fix inconsistent categories (e.g., casing)
   - trim whitespace

6. Drop unnecessary columns
   - placeholder, will evaluate after
  

7. Validate data
   - ensure key fields are not null
   - remove invalid records

## Initiate Data Exploration

In [113]:
#Read available datasets available for data source
DataSources = get_sources()
print(DataSources) #


['311-service-requests-customer-initiated', 'current-and-future-climate', 'neighbourhoods', 'ward-profiles-25-ward-model', 'watermain-breaks', 'watermains']


In [114]:
source = (DataSources[2])
readable = read_files(source)
readable

[WindowsPath('C:/Users/1/Desktop/project/Data/RawData/neighbourhoods/Neighbourhoods - historical 140 fields.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/neighbourhoods/Neighbourhoods fields.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/neighbourhoods/neighbourhoods-2952.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/neighbourhoods/neighbourhoods-4326.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/neighbourhoods/neighbourhoods-historical-140-2952.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/neighbourhoods/neighbourhoods-historical-140-4326.csv')]

### Pull Data to view datasets and keys of datasets from source

In [116]:
neighbourhood = (pull_data(source))
print(f"number of datasets from source: {len(neighbourhood)}")
print(f"names of remaining datasets: {neighbourhood.keys()}")

number of datasets from source: 6
names of remaining datasets: dict_keys(['Neighbourhoods - historical 140 fields', 'Neighbourhoods fields', 'neighbourhoods-2952', 'neighbourhoods-4326', 'neighbourhoods-historical-140-2952', 'neighbourhoods-historical-140-4326'])


### Run summary of the datasets for source

full_summary(data) for a full summary of all datasets, quick_summary(data) for source of singular dataset

since we only have one dataset from source, we do not need to run full summary

In [117]:
full_summary(neighbourhood)
#quick_summary(data_climate['climate-variables'])


Summary for: Neighbourhoods - historical 140 fields
        dtype  missing_pct  n_unique
field  object          0.0        11
name   object          0.0        11

Summary for: Neighbourhoods fields
        dtype  missing_pct  n_unique
field  object          0.0        11
name   object          0.0        11

Summary for: neighbourhoods-2952
                       dtype  missing_pct  n_unique
_id                    int64     0.000000       158
AREA_ID                int64     0.000000       158
AREA_ATTR_ID           int64     0.000000       158
PARENT_AREA_ID       float64     1.000000         0
AREA_SHORT_CODE        int64     0.000000       158
AREA_LONG_CODE         int64     0.000000       158
AREA_NAME             object     0.000000       158
AREA_DESC             object     0.000000       158
CLASSIFICATION        object     0.000000         3
CLASSIFICATION_CODE   object     0.727848         2
OBJECTID               int64     0.000000       158
geometry              object   

### Compare columns for a quick view of how uniform the data is across source

In [119]:
compare_columns(neighbourhood)

,Neighbourhoods - historical 140 fields,Neighbourhoods fields,neighbourhoods-2952,neighbourhoods-4326,neighbourhoods-historical-140-2952,neighbourhoods-historical-140-4326
AREA_ATTR_ID,False,False,True,True,True,True
AREA_DESC,False,False,True,True,True,True
AREA_ID,False,False,True,True,True,True
AREA_LONG_CODE,False,False,True,True,True,True
AREA_NAME,False,False,True,True,True,True
AREA_SHORT_CODE,False,False,True,True,True,True
CLASSIFICATION,False,False,True,True,True,True
CLASSIFICATION_CODE,False,False,True,True,True,True
OBJECTID,False,False,True,True,True,True
PARENT_AREA_ID,False,False,True,True,True,True


### Check for how much presence columns have across all datasets

In [120]:
#check the unique values for a specific column
column_presence(neighbourhood)

,column,presence_score,present_in_tables
0,AREA_ATTR_ID,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."
1,AREA_DESC,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."
2,AREA_ID,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."
3,AREA_LONG_CODE,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."
4,AREA_NAME,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."
5,AREA_SHORT_CODE,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."
6,CLASSIFICATION,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."
7,CLASSIFICATION_CODE,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."
8,OBJECTID,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."
9,PARENT_AREA_ID,0.666667,"[neighbourhoods-2952, neighbourhoods-4326, nei..."


In [97]:
wards_data['2023-wardprofiles-2011-2021-censusdata_rev0719']

,City of Toronto Profiles,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26
0,City of Toronto: City Wards,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021 Census,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Source:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Statistics Canada, 2021 Census, Custom Tabulat...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476,Canada Pension Plan (CPP) and QuÚbec Pensi...,3.5,5.2,3.7,2.8,2.5,5.1,4.5,5.8,2.7,3.4,1.3,1.9,2.3,2.4,2.9,1.8,4.7,4.3,3.6,2.8,4.2,5.3,7.2,7.1,5.5,4.9
1477,Old Age Security pension (OAS) and Guarant...,2.9,3.8,4.2,3,2.3,4,3.4,3.6,2.8,2.5,1,2.2,2.6,1.8,2.3,2.2,4,3.5,2.6,2.8,3.5,4.1,4.3,3.5,3.9,4.2
1478,Employment Insurance (EI) benefits %,1.3,2.1,1.1,1.3,1.2,2.2,1.8,2.5,1,1.6,0.9,0.7,0.9,1.2,1.2,0.6,1.4,1.3,1.1,1.3,1.8,1.9,1.7,1.9,1.8,1.7
1479,Child benefits $,2.8,6.9,2.2,1.6,1.4,5.4,3.7,7.8,1.8,2.1,0.8,0.5,0.9,2,1.7,1.8,4.2,2.8,2.1,2.4,5,5.7,4.3,5.8,6.6,4.2


In [84]:
#the fields seem to be outliers, let's see what it looks like
neighbourhoods_data['neighbourhoods-2952'] #safe to remove fields datasets seems to be just a reference

,_id,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID,geometry
0,1,2502366,26022881,NaN,174,174,South Eglinton-Davisville,South Eglinton-Davisville (174),Not an NIA or Emerging Neighbourhood,NaN,17824737,"{""coordinates"": [[[[313960.56800014555, 483977..."
1,2,2502365,26022880,NaN,173,173,North Toronto,North Toronto (173),Not an NIA or Emerging Neighbourhood,NaN,17824753,"{""coordinates"": [[[[313065.7730001324, 4840780..."
2,3,2502364,26022879,NaN,172,172,Dovercourt Village,Dovercourt Village (172),Not an NIA or Emerging Neighbourhood,NaN,17824769,"{""coordinates"": [[[[310114.0720000863, 4835579..."
3,4,2502363,26022878,NaN,171,171,Junction-Wallace Emerson,Junction-Wallace Emerson (171),Not an NIA or Emerging Neighbourhood,NaN,17824785,"{""coordinates"": [[[[309743.5060000795, 4836414..."
4,5,2502362,26022877,NaN,170,170,Yonge-Bay Corridor,Yonge-Bay Corridor (170),Not an NIA or Emerging Neighbourhood,NaN,17824801,"{""coordinates"": [[[[314155.0620001461, 4833897..."
...,...,...,...,...,...,...,...,...,...,...,...,...
153,154,2502213,26022728,NaN,1,1,West Humber-Clairville,West Humber-Clairville (1),Not an NIA or Emerging Neighbourhood,NaN,17827185,"{""coordinates"": [[[[297520.09499989, 4843788.0..."
154,155,2502212,26022727,NaN,24,24,Black Creek,Black Creek (24),Neighbourhood Improvement Area,NIA,17827201,"{""coordinates"": [[[[303258.7199999766, 4848225..."
155,156,2502211,26022726,NaN,23,23,Pelmo Park-Humberlea,Pelmo Park-Humberlea (23),Not an NIA or Emerging Neighbourhood,NaN,17827217,"{""coordinates"": [[[[302202.0359999621, 4843899..."
156,157,2502210,26022725,NaN,22,22,Humbermede,Humbermede (22),Neighbourhood Improvement Area,NIA,17827233,"{""coordinates"": [[[[302534.63099996175, 484492..."


In [89]:
neighbourhoods_data['neighbourhoods-2952']['CLASSIFICATION'].value_counts()

CLASSIFICATION
Not an NIA or Emerging Neighbourhood    115
Neighbourhood Improvement Area           33
Emerging Neighbourhood                   10
Name: count, dtype: int64

# Toronto Ward Profiles Data

## Description
The 2021 Ward Profiles based on the 25-Ward model (effective December 1, 2018) are available from City Planning. These workbooks contain the data the Profiles are based on, as well as the data the 2016 Ward Profiles (25-Ward Model) were based on. Data from the 2011 National Household Survey is not included.

The 2021 Census Profiles contain Census data from the 2021, 2016 and 2011 Census of Population, including demographic, social and economic information for each Ward in the City of Toronto.

Each Ward Profile provides a snapshot of the population and households in the Ward. The 2021 Ward Profiles contain information on:

    Population
    Occupied Private Dwellings
    Population in Dwellings
    Households
    Families
    Migration and Mobility
    Indigenous Peoples
    Ethnocultural Composition
    Language
    Education
    Labour Force
    Income and Shelter Costs

The 2021 Ward Profiles also include select 2016 and 2011 Census data for comparison purposes. Information from the 2011 Census, from before the 25-Ward model existed, has been adjusted to the current Wards to enable a better understanding of recent population trends and changes.

## Source
City Open Data Catalogue - CKAN API 
https://open.toronto.ca/dataset/ward-profiles-25-ward-model/ Index # is 3

## Files included
25-wardnames-numbers.xlsx - contains name of the ward associated with the ward numbers
2023-wardprofiles-2011-2021-censusdata_rev0719.xlsx - contains census data of each ward

## Files Excluded
2023-wardprofiles-geographicareas.xlsx

## Cleaning Rules -- given the complexity of this database, will return later.
Separate tables in 2023-wardprofiles-2011-2021-censusdata_rev0719, create a dataframe for each one, will have to decide on bucket size for 311 requests as data only spans to 2021 census. Possibly will have to label 2024-2026 as forecast until official data is released using 2021 census.

Likely will exclude geographicareas as a viable dataset, join wardnames with ward-numbers in the future to reduce number of datasets

## Initiate Data Exploration

In [91]:
#Read available datasets available for data source
DataSources = get_sources()
print(DataSources) #


['311-service-requests-customer-initiated', 'current-and-future-climate', 'neighbourhoods', 'ward-profiles-25-ward-model', 'watermain-breaks', 'watermains']


In [92]:
source = (DataSources[3])
readable = read_files(source)
readable

[WindowsPath('C:/Users/1/Desktop/project/Data/RawData/ward-profiles-25-ward-model/2023-wardprofiles-2011-2021-censusdata_rev0719.xlsx'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/ward-profiles-25-ward-model/2023-wardprofiles-geographicareas.xlsx'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/ward-profiles-25-ward-model/25-wardnames-numbers.xlsx')]

### Pull Data to view datasets and keys of datasets from source

In [93]:
wards_data = (pull_data(source))
print(f"number of datasets from source: {len(wards_data)}")
print(f"names of remaining datasets: {wards_data.keys()}")

number of datasets from source: 3
names of remaining datasets: dict_keys(['2023-wardprofiles-2011-2021-censusdata_rev0719', '2023-wardprofiles-geographicareas', '25-wardnames-numbers'])


### Run summary of the datasets for source

full_summary(data) for a full summary of all datasets, quick_summary(data) for source of singular dataset

since we only have one dataset from source, we do not need to run full summary

In [94]:
full_summary(wards_data)
#quick_summary(data_climate['climate-variables'])


Summary for: 2023-wardprofiles-2011-2021-censusdata_rev0719
                           dtype  missing_pct  n_unique
City of Toronto Profiles  object     0.053342       984
Unnamed: 1                object     0.046590       892
Unnamed: 2                object     0.046590       505
Unnamed: 3                object     0.046590       509
Unnamed: 4                object     0.046590       533
Unnamed: 5                object     0.046590       495
Unnamed: 6                object     0.046590       493
Unnamed: 7                object     0.046590       492
Unnamed: 8                object     0.046590       501
Unnamed: 9                object     0.046590       492
Unnamed: 10               object     0.046590       480
Unnamed: 11               object     0.046590       519
Unnamed: 12               object     0.046590       494
Unnamed: 13               object     0.046590       516
Unnamed: 14               object     0.046590       527
Unnamed: 15               object     0.0465

### Compare columns for a quick view of how uniform the data is across source

In [95]:
compare_columns(wards_data)

,2023-wardprofiles-2011-2021-censusdata_rev0719,2023-wardprofiles-geographicareas,25-wardnames-numbers
City of Toronto Profiles,True,True,False
Unnamed: 1,True,True,False
Unnamed: 10,True,False,False
Unnamed: 11,True,False,False
Unnamed: 12,True,False,False
Unnamed: 13,True,False,False
Unnamed: 14,True,False,False
Unnamed: 15,True,False,False
Unnamed: 16,True,False,False
Unnamed: 17,True,False,False


### Check for how much presence columns have across all datasets

In [96]:
#check the unique values for a specific column
column_presence(wards_data)

,column,presence_score,present_in_tables
0,City of Toronto Profiles,0.666667,[2023-wardprofiles-2011-2021-censusdata_rev071...
1,Unnamed: 1,0.666667,[2023-wardprofiles-2011-2021-censusdata_rev071...
2,Unnamed: 10,0.333333,[2023-wardprofiles-2011-2021-censusdata_rev0719]
3,Unnamed: 11,0.333333,[2023-wardprofiles-2011-2021-censusdata_rev0719]
4,Unnamed: 12,0.333333,[2023-wardprofiles-2011-2021-censusdata_rev0719]
5,Unnamed: 13,0.333333,[2023-wardprofiles-2011-2021-censusdata_rev0719]
6,Unnamed: 14,0.333333,[2023-wardprofiles-2011-2021-censusdata_rev0719]
7,Unnamed: 15,0.333333,[2023-wardprofiles-2011-2021-censusdata_rev0719]
8,Unnamed: 16,0.333333,[2023-wardprofiles-2011-2021-censusdata_rev0719]
9,Unnamed: 17,0.333333,[2023-wardprofiles-2011-2021-censusdata_rev0719]


In [97]:
wards_data['2023-wardprofiles-2011-2021-censusdata_rev0719']

,City of Toronto Profiles,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26
0,City of Toronto: City Wards,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021 Census,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Source:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Statistics Canada, 2021 Census, Custom Tabulat...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476,Canada Pension Plan (CPP) and QuÚbec Pensi...,3.5,5.2,3.7,2.8,2.5,5.1,4.5,5.8,2.7,3.4,1.3,1.9,2.3,2.4,2.9,1.8,4.7,4.3,3.6,2.8,4.2,5.3,7.2,7.1,5.5,4.9
1477,Old Age Security pension (OAS) and Guarant...,2.9,3.8,4.2,3,2.3,4,3.4,3.6,2.8,2.5,1,2.2,2.6,1.8,2.3,2.2,4,3.5,2.6,2.8,3.5,4.1,4.3,3.5,3.9,4.2
1478,Employment Insurance (EI) benefits %,1.3,2.1,1.1,1.3,1.2,2.2,1.8,2.5,1,1.6,0.9,0.7,0.9,1.2,1.2,0.6,1.4,1.3,1.1,1.3,1.8,1.9,1.7,1.9,1.8,1.7
1479,Child benefits $,2.8,6.9,2.2,1.6,1.4,5.4,3.7,7.8,1.8,2.1,0.8,0.5,0.9,2,1.7,1.8,4.2,2.8,2.1,2.4,5,5.7,4.3,5.8,6.6,4.2


In [84]:
#the fields seem to be outliers, let's see what it looks like
neighbourhoods_data['neighbourhoods-2952'] #safe to remove fields datasets seems to be just a reference

,_id,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID,geometry
0,1,2502366,26022881,NaN,174,174,South Eglinton-Davisville,South Eglinton-Davisville (174),Not an NIA or Emerging Neighbourhood,NaN,17824737,"{""coordinates"": [[[[313960.56800014555, 483977..."
1,2,2502365,26022880,NaN,173,173,North Toronto,North Toronto (173),Not an NIA or Emerging Neighbourhood,NaN,17824753,"{""coordinates"": [[[[313065.7730001324, 4840780..."
2,3,2502364,26022879,NaN,172,172,Dovercourt Village,Dovercourt Village (172),Not an NIA or Emerging Neighbourhood,NaN,17824769,"{""coordinates"": [[[[310114.0720000863, 4835579..."
3,4,2502363,26022878,NaN,171,171,Junction-Wallace Emerson,Junction-Wallace Emerson (171),Not an NIA or Emerging Neighbourhood,NaN,17824785,"{""coordinates"": [[[[309743.5060000795, 4836414..."
4,5,2502362,26022877,NaN,170,170,Yonge-Bay Corridor,Yonge-Bay Corridor (170),Not an NIA or Emerging Neighbourhood,NaN,17824801,"{""coordinates"": [[[[314155.0620001461, 4833897..."
...,...,...,...,...,...,...,...,...,...,...,...,...
153,154,2502213,26022728,NaN,1,1,West Humber-Clairville,West Humber-Clairville (1),Not an NIA or Emerging Neighbourhood,NaN,17827185,"{""coordinates"": [[[[297520.09499989, 4843788.0..."
154,155,2502212,26022727,NaN,24,24,Black Creek,Black Creek (24),Neighbourhood Improvement Area,NIA,17827201,"{""coordinates"": [[[[303258.7199999766, 4848225..."
155,156,2502211,26022726,NaN,23,23,Pelmo Park-Humberlea,Pelmo Park-Humberlea (23),Not an NIA or Emerging Neighbourhood,NaN,17827217,"{""coordinates"": [[[[302202.0359999621, 4843899..."
156,157,2502210,26022725,NaN,22,22,Humbermede,Humbermede (22),Neighbourhood Improvement Area,NIA,17827233,"{""coordinates"": [[[[302534.63099996175, 484492..."


In [89]:
neighbourhoods_data['neighbourhoods-2952']['CLASSIFICATION'].value_counts()

CLASSIFICATION
Not an NIA or Emerging Neighbourhood    115
Neighbourhood Improvement Area           33
Emerging Neighbourhood                   10
Name: count, dtype: int64

# Dataset: Watermain-Breaks
Index 0 on DataSources


## Description
BREAKDATE - Date of watermain break reported BREAKYEAR - Year of watermain break reported
XCOORD - Easting in MTM NAD 27(3 degree) Projection & Lat and Long
YCOORD - Northing in MTMNAD 27(3 degree) Projection & Lat and Long

This dataset is used to track watermain breaks within the City of Toronto boundaries. It contains the date and location (X and Y coordinates) of watermain breaks over a 27 year period from 1990-2016

FID 		- Sequential unique whole numbers that are automatically generated.
SHAPE 		- Coordinates defining the features.
BREAK_DATE	- Date watermain break reported
BREAK_YEAR	- Year watermain break reported
XCOORD		- Easting in MTM NAD27 3 degree Projection
YCOORD		- Northing in MTM NAD27 3 degree Projection
POINT_X		- Longitude in WGS84 Coordinate System
POINT_Y		- Latitude in WGS84 Coordinate System

## Source
City Open Data Catalogue - CKAN API 
https://open.toronto.ca/dataset/watermain-breaks/ Index # is 4

## Files included
Watermain Breaks WGS84 Read_me_file.txt
watermain-breaks-1990-to-2016-excel.xlsx

## Files excluded
all shape files as we will be using the coordinates in excel

## Grain
One instance of a 311 service request

## Cleaning Rules

1. Remove non-data files (e.g., readme)

2. Standardize column names
   - convert to snake_case
   - lowercase, remove spaces/special characters

3. Align schema across datasets
   - ensure consistent column structure before combining

4. Handle missing values
   - assess missingness
   - decide to drop, impute, or retain

5. Enforce data types
   - convert date fields to datetime
   - ensure numeric fields are numeric
   - cast categorical fields appropriately

6. Standardize values
   - fix inconsistent categories (e.g., casing)
   - trim whitespace

7. Drop unnecessary columns
   - Remove all but Creation_date, Status, Postal, Ward, Type, Division, Section

8. Validate data
   - ensure key fields are not null
   - remove invalid records


## Initiate Data Exploration

In [109]:
#Read available datasets available for data source
DataSources = get_sources()
print(DataSources)

['311-service-requests-customer-initiated', 'current-and-future-climate', 'neighbourhoods', 'ward-profiles-25-ward-model', 'watermain-breaks', 'watermains']


In [99]:
source = (DataSources[4])
readable = read_files(source)
readable

[WindowsPath('C:/Users/1/Desktop/project/Data/RawData/watermain-breaks/watermain-breaks-1990-to-2016-excel.xlsx')]

### Pull Data to view datasets and keys of datasets from source

In [104]:
breaks_data = (pull_data(source))
print(f"number of datasets from source: {len(breaks_data)}")
print(f"names of remaining datasets: {breaks_data.keys()}")

number of datasets from source: 1
names of remaining datasets: dict_keys(['watermain-breaks-1990-to-2016-excel'])


### Run summary of the datasets for source

full_summary(data) for a full summary of all datasets, quick_summary(data) for source of singular dataset

In [105]:
full_summary(breaks_data)
#quick_summary(data_311['SR2026'])


Summary for: watermain-breaks-1990-to-2016-excel
                     dtype  missing_pct  n_unique
BREAK DATE  datetime64[ns]          0.0      8219
BREAK YEAR           int64          0.0        27
X COORD            float64          0.0     35166
Y COORD            float64          0.0     35151


In [108]:
breaks_data['watermain-breaks-1990-to-2016-excel'].head()

,BREAK DATE,BREAK YEAR,X COORD,Y COORD
0,1990-01-01,1990,301316.434,4829245.578
1,1990-01-01,1990,326711.060,4846900.024
2,1990-01-01,1990,307098.866,4841152.120
3,1990-01-01,1990,306976.247,4842116.883
4,1990-01-01,1990,326543.178,4847968.844


# Dataset: watermains


## Description
This dataset provides pertinent information on the location of water system pipes, watermain type based on function, and pipe characteristics (e.g. diameter, material etc.). The data is updated on a continual daily basis. The data is intended to support development and growth by facilitating pre-application discussions and preparation of servicing reports by Developers (and their teams).

## Source
City Open Data Catalogue - CKAN API 
https://open.toronto.ca/dataset/watermains/ Index # is 5

## Files included

distribution-watermain-2952.csv
distribution-watermain-4326.csv
transmission-watermain-2952.csv
transmission-watermain-4326.csv

## Files excluded
Distribution Watermain fields.csv
distribution-watermain shape files
transmission-watermain shape files

## Cleaning Rules

1. Remove non-data files (e.g., readme)

2. Standardize column names
   - convert to snake_case
   - lowercase, remove spaces/special characters

3. Align schema across datasets
   - ensure consistent column structure before combining

4. Handle missing values
   - assess missingness
   - decide to drop, impute, or retain

5. Enforce data types
   - convert date fields to datetime
   - ensure numeric fields are numeric
   - cast categorical fields appropriately

6. Standardize values
   - fix inconsistent categories (e.g., casing)
   - trim whitespace

7. Drop unnecessary columns
   - Remove all but Creation_date, Status, Postal, Ward, Type, Division, Section

8. Validate data
   - ensure key fields are not null
   - remove invalid records


## Initiate Data Exploration

In [123]:
#Read available datasets available for data source
DataSources = get_sources()
print(DataSources)

['311-service-requests-customer-initiated', 'current-and-future-climate', 'neighbourhoods', 'ward-profiles-25-ward-model', 'watermain-breaks', 'watermains']


In [122]:
source = (DataSources[5])
readable = read_files(source)
readable

[WindowsPath('C:/Users/1/Desktop/project/Data/RawData/watermains/Distribution Watermain fields.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/watermains/distribution-watermain-2952.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/watermains/distribution-watermain-4326.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/watermains/Transmission Watermain fields.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/watermains/transmission-watermain-2952.csv'),
 WindowsPath('C:/Users/1/Desktop/project/Data/RawData/watermains/transmission-watermain-4326.csv')]

### Pull Data to view datasets and keys of datasets from source

In [142]:
watermain = (pull_data(source))
print(f"number of datasets from source: {len(watermain)}")
print(f"names of remaining datasets: {watermain.keys()}")

number of datasets from source: 6
names of remaining datasets: dict_keys(['Distribution Watermain fields', 'distribution-watermain-2952', 'distribution-watermain-4326', 'Transmission Watermain fields', 'transmission-watermain-2952', 'transmission-watermain-4326'])


### Run summary of the datasets for source

full_summary(data) for a full summary of all datasets, quick_summary(data) for source of singular dataset

In [131]:
full_summary(watermain)
quick_summary(watermain['distribution-watermain-2952'])


Summary for: Distribution Watermain fields
        dtype  missing_pct  n_unique
field  object          0.0         9
name   object          0.0         9

Summary for: distribution-watermain-2952
                                  dtype  missing_pct  n_unique
_id                               int64     0.000000     46624
Watermain Asset Identification   object     0.000000     46624
Watermain Type                    int64     0.000000         1
Watermain Diameter              float64     0.000922        21
Watermain Material               object     0.000300        17
Watermain Install Date           object     0.016601       299
Watermain Construction Year     float64     0.016622       153
Watermain Measured Length       float64     0.002702     16242
Watermain Location Description   object     0.000493     10010
geometry                         object     0.000000     46623

Summary for: distribution-watermain-4326
                                  dtype  missing_pct  n_unique
_id  

,dtype,missing_pct,n_unique
_id,int64,0.000000,46624
Watermain Asset Identification,object,0.000000,46624
Watermain Type,int64,0.000000,1
Watermain Diameter,float64,0.000922,21
Watermain Material,object,0.000300,17
Watermain Install Date,object,0.016601,299
Watermain Construction Year,float64,0.016622,153
Watermain Measured Length,float64,0.002702,16242
Watermain Location Description,object,0.000493,10010
geometry,object,0.000000,46623


### Compare columns for a quick view of how uniform the data is across source

In [127]:
compare_columns(watermain)

,Distribution Watermain fields,distribution-watermain-2952,distribution-watermain-4326,Transmission Watermain fields,transmission-watermain-2952,transmission-watermain-4326
Watermain Asset Identification,False,True,True,False,True,True
Watermain Construction Year,False,True,True,False,True,True
Watermain Diameter,False,True,True,False,True,True
Watermain Install Date,False,True,True,False,True,True
Watermain Location Description,False,True,True,False,True,True
Watermain Material,False,True,True,False,True,True
Watermain Measured Length,False,True,True,False,True,True
Watermain Type,False,True,True,False,True,True
_id,False,True,True,False,True,True
field,True,False,False,True,False,False


### Check for how much presence columns have across all datasets

In [128]:
column_presence(watermain)
#check the unique values for a specific column

,column,presence_score,present_in_tables
0,Watermain Asset Identification,0.666667,"[distribution-watermain-2952, distribution-wat..."
1,Watermain Construction Year,0.666667,"[distribution-watermain-2952, distribution-wat..."
2,Watermain Diameter,0.666667,"[distribution-watermain-2952, distribution-wat..."
3,Watermain Install Date,0.666667,"[distribution-watermain-2952, distribution-wat..."
4,Watermain Location Description,0.666667,"[distribution-watermain-2952, distribution-wat..."
5,Watermain Material,0.666667,"[distribution-watermain-2952, distribution-wat..."
6,Watermain Measured Length,0.666667,"[distribution-watermain-2952, distribution-wat..."
7,Watermain Type,0.666667,"[distribution-watermain-2952, distribution-wat..."
8,_id,0.666667,"[distribution-watermain-2952, distribution-wat..."
9,geometry,0.666667,"[distribution-watermain-2952, distribution-wat..."


In [145]:
summaryd = watermain['distribution-watermain-2952']
summaryd[summaryd['Watermain Install Date'].isnull()]

,_id,Watermain Asset Identification,Watermain Type,Watermain Diameter,Watermain Material,Watermain Install Date,Watermain Construction Year,Watermain Measured Length,Watermain Location Description,geometry
39,40,LN101181,0,150.0,DIP,NaN,NaN,356.563981,COVERDALE CRES,"{""coordinates"": [[[319939.6720002378, 4853189...."
107,108,LN2444903,0,200.0,UNK,NaN,NaN,65.900000,MARTIN GROVE RD,"{""coordinates"": [[[298698.6659999025, 4840261...."
176,177,LN2400277,0,300.0,PVC,NaN,NaN,95.100000,MARINE PARADE DR,"{""coordinates"": [[[306328.62400002155, 4831446..."
207,208,LN108438,0,150.0,UNK,NaN,NaN,21.200000,MANSE RD,"{""coordinates"": [[[330961.7820004085, 4847575...."
294,295,LN102063,0,150.0,PVC,NaN,NaN,240.400000,WILDERNESS DR,"{""coordinates"": [[[322817.2540002825, 4854376...."
...,...,...,...,...,...,...,...,...,...,...
46274,46275,LN63549,0,200.0,NaN,NaN,NaN,13.040000,NaN,"{""coordinates"": [[[311335.32000009913, 4833080..."
46275,46276,LN63546-2,0,200.0,NaN,NaN,NaN,58.440000,Hanna Ave,"{""coordinates"": [[[311342.83900010085, 4833080..."
46276,46277,LN63546-1,0,200.0,NaN,NaN,NaN,8.110000,Hanna Ave,"{""coordinates"": [[[311335.32000009913, 4833080..."
46291,46292,LN63563,0,150.0,UNK,NaN,NaN,3.830000,GEORGE ST,"{""coordinates"": [[[302998.05699997593, 4840412..."
